In [1]:
import os
import time
import joblib
import numpy as np
import pandas as pd
import clickhouse_connect
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

## 1. KONFIGURASI & KONEKSI

In [2]:
print("🔄 Menghubungkan ke AWS ClickHouse Cluster...")

client = clickhouse_connect.get_client(
    host="13.215.79.3", 
    port=8123,
    username="default",
    password="rahasia123",
    database="flight_delay"
)

# Definisikan kolom fitur sesuai dokumen panduan kelompok
FEATURE_COLS = [
    'flight_month', 'dep_hour', 'is_weekend', 'season',
    'distance_bucket', 'same_state_route',
    'route_avg_arr_delay_prev', 'carrier_arr_delay_rate_prev'
]
TARGET_COL = 'arr_del15_label'

🔄 Menghubungkan ke AWS ClickHouse Cluster...


## 2. FUNGSI PREPROCESSING PER CHUNK

In [3]:
def preprocess_chunk(df_chunk, encoders=None, is_training=True):
    """Melakukan handling missing value, encoding, dan BALANCING DATA (Undersampling)"""
    
    # 1. ISI NILAI KOSONG & DOWNCASTING (Sama seperti sebelumnya)
    int_cols_to_fill = ['flight_month', 'dep_hour', 'is_weekend', 'same_state_route']
    for col in int_cols_to_fill:
        if col in df_chunk.columns:
            df_chunk[col] = df_chunk[col].fillna(0).astype(np.int8)

    if 'route_avg_arr_delay_prev' in df_chunk.columns:
        df_chunk['route_avg_arr_delay_prev'] = df_chunk['route_avg_arr_delay_prev'].fillna(0)
    if 'carrier_arr_delay_rate_prev' in df_chunk.columns:
        df_chunk['carrier_arr_delay_rate_prev'] = df_chunk['carrier_arr_delay_rate_prev'].fillna(0)
        
    if TARGET_COL in df_chunk.columns:
        df_chunk[TARGET_COL] = df_chunk[TARGET_COL].fillna(0).astype(np.int8)

    # 2. LABEL ENCODING KOLOM KATEGORIKAL
    categorical_cols = ['season', 'distance_bucket']
    if encoders is None: encoders = {}

    for col in categorical_cols:
        if col in df_chunk.columns:
            df_chunk[col] = df_chunk[col].fillna('Unknown').astype(str)
            if is_training:
                le = LabelEncoder()
                df_chunk[col] = le.fit_transform(df_chunk[col])
                encoders[col] = le
            else:
                le = encoders[col]
                df_chunk[col] = df_chunk[col].map(lambda s: s if s in le.classes_ else le.classes_[0])
                df_chunk[col] = le.transform(df_chunk[col])
                
    # 3. 🔥 PROSES MENYAMAKAN RASIO DATA (Hanya dilakukan saat TRAINING)
    if is_training and TARGET_COL in df_chunk.columns:
        # Pisahkan kelas 0 dan 1
        df_ontime = df_chunk[df_chunk[TARGET_COL] == 0]
        df_delay = df_chunk[df_chunk[TARGET_COL] == 1]
        
        n_delay = len(df_delay)
        
        # Jaga-jaga jika ada batch yang tidak memiliki data delay sama sekali
        if n_delay > 0 and len(df_ontime) > n_delay:
            # Ambil data ontime secara acak sebanyak jumlah data delay
            df_ontime_sampled = df_ontime.sample(n=n_delay, random_state=42)
            # Gabungkan kembali data yang sudah seimbang
            df_chunk = pd.concat([df_ontime_sampled, df_delay], ignore_index=True)
            # Acak urutannya agar model tidak belajar polanya berurutan
            df_chunk = df_chunk.sample(frac=1, random_state=42).reset_index(drop=True)

    return df_chunk, encoders

## 3. PROSES STREAMING & TRAINING BATCH (Incremental Learning)

In [4]:
# Karena data berskala Big Data, kita gunakan SGDClassifier yang mendukung .partial_fit()
# Ini memungkinkan kita melatih model per chunk tanpa meload seluruh data ke RAM sekaligus.
models = {
    "Logistic Regression": SGDClassifier(loss='log_loss', random_state=42),
    "Linear SVM": SGDClassifier(loss='hinge', random_state=42),
    "Passive Aggressive": SGDClassifier(loss='hinge', penalty=None, learning_rate='pa1', eta0=1.0, random_state=42)
}

encoders = None

query_train = f"""
    SELECT {', '.join(FEATURE_COLS)}, {TARGET_COL} 
    FROM ontime_features 
    WHERE FlightDate < '2025-01-01'
"""

print("\n🚀 Memulai Ingestion & Incremental Training (Data 2021-2024)...")
try:
    # Membuka stream block native dari ClickHouse
    stream = client.query_column_block_stream(query_train)
    chunk_idx = 0
    total_rows_trained = 0
    
    with stream:
        for block in stream:
            chunk_idx += 1
            try:
                # Konversi block ke DataFrame
                df_chunk = pd.DataFrame(dict(zip(stream.source.column_names, block)))
                if df_chunk.empty:
                    continue
                
                # Preprocess chunk saat ini
                df_chunk, encoders = preprocess_chunk(df_chunk, encoders, is_training=True)
                
                X_chunk = df_chunk[FEATURE_COLS]
                y_chunk = df_chunk[TARGET_COL]
                
                classes = np.array([0, 1])
                
                # Jika pesawat Delay (1), beri bobot/hukuman 10x lipat. Jika Tepat Waktu (0), beri bobot normal 1.
                # Kamu bisa menaikkan angka 10.0 menjadi 15.0 atau 20.0 jika hasil recall nanti dirasa masih kurang greget.
                sample_weights = np.where(y_chunk == 1, 10.0, 1.0)
                
                # Latih ke-3 model secara bersamaan pada chunk ini
                for name, model in models.items():
                    model.partial_fit(X_chunk, y_chunk, classes=classes, sample_weight=sample_weights)
                
                total_rows_trained += len(df_chunk)
                
                if chunk_idx % 5 == 0 or chunk_idx == 1:
                    print(f"   [Batch #{chunk_idx:03d}] Berhasil melatih {len(df_chunk):,} baris data ke 3 Algoritma. (Total sementara: {total_rows_trained:,})")
                    
            except Exception as e:
                print(f"   ❌ Gagal memproses Batch Training #{chunk_idx}: {str(e)}")
                continue

    print("✅ Proses Training Selesai Berdampingan dengan Stream!")
    print(f"📈 TOTAL DATA DITRAINING: {total_rows_trained:,} baris.")

except Exception as fatal_err:
    print(f"💥 Kegagalan Sistem Stream Training: {str(fatal_err)}")


🚀 Memulai Ingestion & Incremental Training (Data 2021-2024)...
   [Batch #001] Berhasil melatih 4,424 baris data ke 3 Algoritma. (Total sementara: 4,424)
   [Batch #005] Berhasil melatih 6,090 baris data ke 3 Algoritma. (Total sementara: 31,094)
   [Batch #010] Berhasil melatih 8,606 baris data ke 3 Algoritma. (Total sementara: 67,822)
   [Batch #015] Berhasil melatih 4,440 baris data ke 3 Algoritma. (Total sementara: 95,892)
   [Batch #020] Berhasil melatih 6,462 baris data ke 3 Algoritma. (Total sementara: 136,204)
   [Batch #025] Berhasil melatih 13,842 baris data ke 3 Algoritma. (Total sementara: 182,198)
   [Batch #030] Berhasil melatih 8,140 baris data ke 3 Algoritma. (Total sementara: 222,618)
   [Batch #035] Berhasil melatih 7,348 baris data ke 3 Algoritma. (Total sementara: 260,640)
   [Batch #040] Berhasil melatih 13,386 baris data ke 3 Algoritma. (Total sementara: 312,920)
   [Batch #045] Berhasil melatih 3,028 baris data ke 3 Algoritma. (Total sementara: 353,972)
   [Batch

## 4. PROSES STREAMING TESTING & EVALUASI 3 MODEL

In [5]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import pandas as pd

# Pastikan variabel query_test sudah didefinisikan
query_test = f"""
    SELECT {', '.join(FEATURE_COLS)}, {TARGET_COL} 
    FROM ontime_features 
    WHERE FlightDate >= '2025-01-01'
"""

print("\n🕵️ Memulai Evaluasi Model pada Data Testing (Data 2025)...")
y_true_all = []

# Siapkan wadah untuk prediksi biner (0/1) dan skor keputusan (untuk ROC-AUC)
y_pred_all = {name: [] for name in models.keys()}
y_score_all = {name: [] for name in models.keys()} # 👈 Tambahan wadah untuk skor ROC-AUC

try:
    test_stream = client.query_column_block_stream(query_test)
    test_chunk_idx = 0
    
    with test_stream:
        for block in test_stream:
            test_chunk_idx += 1
            try:
                df_test_chunk = pd.DataFrame(dict(zip(test_stream.source.column_names, block)))
                if df_test_chunk.empty:
                    continue
                
                # Preprocess data testing (is_training=False agar tidak di-undersample)
                df_test_chunk, _ = preprocess_chunk(df_test_chunk, encoders, is_training=False)
                X_test_chunk = df_test_chunk[FEATURE_COLS]
                y_test_chunk = df_test_chunk[TARGET_COL]
                
                # Simpan jawaban asli lapangan
                y_true_all.extend(y_test_chunk.tolist())
                
                # Ambil prediksi dan score dari ke-3 model
                for name, model in models.items():
                    # 1. Prediksi biner untuk Accuracy, Precision, Recall, F1
                    preds = model.predict(X_test_chunk)
                    y_pred_all[name].extend(preds.tolist())
                    
                    # 2. 👈 Ambil skor keputusan mentah khusus untuk kalkulasi ROC-AUC
                    scores = model.decision_function(X_test_chunk)
                    y_score_all[name].extend(scores.tolist())
                
            except Exception as e:
                print(f"   ❌ Gagal memproses Batch Testing #{test_chunk_idx}: {str(e)}")
                continue

    # -------------------------------------------------------------
    # PEMBUATAN TABEL KOMPARASI PERFORMA MODEL (TERMASUK ROC-AUC)
    # -------------------------------------------------------------
    print("\n📊 TABEL EVALUASI AKHIR (PERBANDINGAN 3 MODEL):")
    eval_results = []
    
    for name in models.keys():
        acc = accuracy_score(y_true_all, y_pred_all[name])
        prec = precision_score(y_true_all, y_pred_all[name], zero_division=0)
        rec = recall_score(y_true_all, y_pred_all[name], zero_division=0)
        f1 = f1_score(y_true_all, y_pred_all[name], zero_division=0)
        # 👈 3. Hitung skor ROC-AUC menggunakan skor keputusan mentah
        auc = roc_auc_score(y_true_all, y_score_all[name]) 
        
        eval_results.append({
            "Algoritma": name,
            "Accuracy": f"{acc:.4f}",
            "Precision": f"{prec:.4f}",
            "Recall": f"{rec:.4f}",
            "F1-Score": f"{f1:.4f}",
            "ROC-AUC": f"{auc:.4f}" 
        })
        
    df_eval = pd.DataFrame(eval_results)
    print("=" * 75)
    print(df_eval.to_string(index=False))
    print("=" * 75)

except Exception as fatal_err:
    print(f"💥 Kegagalan Sistem Stream Testing: {str(fatal_err)}")


🕵️ Memulai Evaluasi Model pada Data Testing (Data 2025)...

📊 TABEL EVALUASI AKHIR (PERBANDINGAN 3 MODEL):
          Algoritma Accuracy Precision Recall F1-Score ROC-AUC
Logistic Regression   0.3671    0.2422 0.8943   0.3812  0.6226
         Linear SVM   0.2199    0.2183 0.9992   0.3583  0.6054
 Passive Aggressive   0.6739    0.2759 0.3054   0.2899  0.5731


## 5. MENYIMPAN ARTIFACTS KE DIREKTORI MODELS

In [6]:
print("\n💾 Mengamankan hasil pemodelan ke dalam folder /models...")
os.makedirs('models', exist_ok=True)

for name, model in models.items():
    # Ubah nama "Logistic Regression" menjadi "logistic_regression.pkl"
    file_name = name.lower().replace(" ", "_") + "_best.pkl"
    joblib.dump(model, f'models/{file_name}')
    print(f"   ✔️ Disimpan: {file_name}")

joblib.dump(encoders, 'models/encoders.pkl')
print("\n🎉 Semua proses selesai dengan bersih! File pkl dan tabel siap dikirim.")


💾 Mengamankan hasil pemodelan ke dalam folder /models...
   ✔️ Disimpan: logistic_regression_best.pkl
   ✔️ Disimpan: linear_svm_best.pkl
   ✔️ Disimpan: passive_aggressive_best.pkl

🎉 Semua proses selesai dengan bersih! File pkl dan tabel siap dikirim.
